# BA283 A/B — FiLM vs plain GRU (concat)
Copies the expanded dataset to local disk (rclone copy, no FUSE mount), then trains both conditionings on identical data, compares val ESR, exports both.

## 1. Install rclone + configure remote
No mount: the dataset is copied to `/content/dataset` in a later step so training never reads through a FUSE/VFS layer.

Run the cell, then **upload your `rclone.conf`** when prompted (find it via `rclone config file`). It holds the Drive token, so it's written straight to disk and never printed.

In [ ]:
import os, glob, subprocess
from google.colab import files

# install rclone (no-op if present)
if subprocess.run(['which','rclone'],capture_output=True).returncode:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash', shell=True, check=True)

# upload rclone.conf (token written to disk, not echoed)
cfg = os.path.expanduser('~/.config/rclone/rclone.conf')
os.makedirs(os.path.dirname(cfg), exist_ok=True)
up = files.upload()  # pick your rclone.conf
assert up, 'no file uploaded'
open(cfg,'wb').write(next(iter(up.values())))
REMOTE = subprocess.run(['rclone','listremotes'],capture_output=True,text=True).stdout.split()[0]
print('remote:', REMOTE)

## 2. Get the training code + deps
Clone the repo first so `train.stage` is importable for the copy step below. Paste a GitHub token with read access (kept out of output via getpass).

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ')
import subprocess
subprocess.run(['git','clone','-q','https://x-access-token:%s@github.com/gustavokch/ba283-vst3.git'%TOKEN,'/content/ba283'])
%cd /content/ba283
!git pull -q
!pip -q install soundfile pandas tensorboard scipy
import torch; print('cuda:', torch.cuda.is_available())

## 3. Copy dataset to local disk via rclone
`stage_rclone` runs `rclone copy <remote>:ba283_spice_gain /content/dataset -P` (no mount) and asserts `manifest.csv` is present. Every WAV lands on local disk before training reads it.

In [ ]:
from train.stage import stage_rclone, remap_manifests
DATA = str(stage_rclone(REMOTE + 'ba283_spice_gain', '/content/dataset'))
print('staged ->', DATA)

## 4. Rewrite manifest paths to the local dataset location
The box wrote absolute paths; `remap_manifests` rewrites each to `/content/dataset/<wet|dry>/<basename>` and writes train/val/test.csv to `/content`.

In [ ]:
M = remap_manifests('/content/dataset', out_dir='/content')
print(M)

## 5. Train FiLM

In [ ]:
!python -m train.train --train-manifest /content/train.csv --val-manifest /content/val.csv \
  --epochs 200 --lr 1e-3 --batch 128 --stride 4096 --hidden 32 --device cuda --conditioning film \
  --checkpoint-dir ckpt_film --log-dir logs_film --patience 40 --seed 42

## 5b. Checkpoint FiLM to Drive (runtime-limit insurance)
Run right after FiLM finishes. Evals val ESR, exports the JSON + raw checkpoints, and `rclone copy`s them straight to the remote (synchronous, so they survive a runtime kill before concat finishes).

In [ ]:
import torch, numpy as np, os, glob, shutil, subprocess
from train.dataset import make_loader
from train.model import BA283LSTM
from train.export import export_rtneural_json
from train.loss import esr

def eval_val(cond, ckpt):
    m = BA283LSTM(32, conditioning=cond)
    m.load_state_dict(torch.load(ckpt+'/best.pth', map_location='cpu', weights_only=True)); m.eval()
    ld = make_loader('/content/val.csv', batch_size=32, window=4096, stride=2048,
                     num_workers=2, shuffle=False, drop_last=False)
    P=[];T=[];warm=1024
    with torch.no_grad():
        for x,y in ld:
            _,h=m(x[:,:warm]); p,_=m(x[:,warm:],h)
            P.append(p.reshape(-1)); T.append(y[:,warm:].reshape(-1))
    return m, esr(torch.cat(P),torch.cat(T)).item()

m, e_film = eval_val('film','ckpt_film')
print('film val ESR = %.4f (%.1f dB)' % (e_film, 10*np.log10(max(e_film,1e-12))))

# stage locally, then rclone copy straight to the remote (synchronous upload)
os.makedirs('/content/film_save', exist_ok=True)
export_rtneural_json(m, '/content/film_save/ba283_film.json',
                     meta={'source':'colab-ab','conditioning':'film','val_esr':e_film})
for f in glob.glob('ckpt_film/*.pth'):
    shutil.copy(f, '/content/film_save/film_'+os.path.basename(f))
open('/content/film_save/film_val_esr.txt','w').write('%.6f\n' % e_film)
REMOTE = globals().get('REMOTE','gdrive:')
subprocess.run(['rclone','copy','/content/film_save', REMOTE+'ba283_spice_gain/film_save','-P'], check=True)
print('saved -> %sba283_spice_gain/film_save' % REMOTE)

## 6. Train plain GRU (concat)

In [ ]:
!python -m train.train --train-manifest /content/train.csv --val-manifest /content/val.csv \
  --epochs 200 --lr 1e-3 --batch 128 --stride 4096 --hidden 32 --device cuda --conditioning concat \
  --checkpoint-dir ckpt_concat --log-dir logs_concat --patience 40 --seed 42

## 7. Compare val ESR + export both to Drive

In [ ]:
import torch, numpy as np, os, subprocess
from train.dataset import make_loader
from train.model import BA283LSTM
from train.export import export_rtneural_json
from train.loss import esr
os.makedirs('/content/ab_save', exist_ok=True)
def evalm(cond, ckpt):
    m = BA283LSTM(32, conditioning=cond)
    m.load_state_dict(torch.load(ckpt+'/best.pth', map_location='cpu', weights_only=True)); m.eval()
    ld = make_loader('/content/val.csv', batch_size=32, window=4096, stride=2048, num_workers=2, shuffle=False, drop_last=False)
    P=[];T=[];warm=1024
    with torch.no_grad():
        for x,y in ld:
            _,h=m(x[:,:warm]); p,_=m(x[:,warm:],h); P.append(p.reshape(-1)); T.append(y[:,warm:].reshape(-1))
    e=esr(torch.cat(P),torch.cat(T)).item()
    export_rtneural_json(m, '/content/ab_save/ba283_%s.json'%cond, meta={'source':'colab-ab','conditioning':cond})
    return e
res = {c: evalm(c, ck) for c,ck in [('film','ckpt_film'),('concat','ckpt_concat')]}
for c,e in res.items(): print('%-7s val ESR = %.4f (%.1f dB)' % (c, e, 10*np.log10(max(e,1e-12))))
win = min(res, key=res.get); print('WINNER:', win)
REMOTE = globals().get('REMOTE','gdrive:')
subprocess.run(['rclone','copy','/content/ab_save', REMOTE+'ba283_spice_gain/ab_save','-P'], check=True)
print('exported ba283_film.json + ba283_concat.json -> %sba283_spice_gain/ab_save' % REMOTE)

Bring back `ba283_<winner>.json` (on Drive). I embed it + port that conditioning to the Rust plugin (#30).